# In this notebook we try to diagonalize the fluxonium Hamiltonian (ATS written as fluxonium)

The ATS Hamiltonian reads $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\phi_\Sigma)\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + 2\Delta E_J\sin(\phi_\Sigma)\sin(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta)$$

We can set $\phi_\Sigma = 0$ so that $$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta),$$
we retrieve the Fluxonium Hamiltonian.

Two caveats : 
- experimentally you don't set $\phi_\Sigma = 0$ exactly (few degrees off) and then $\Delta E_J/E_J < 10 \%$
- there is another term $E_{2J}\cos(2\phi_\Sigma)\cos(2\varphi_a (a+a^\dagger) + 2\varphi_b (b+b^\dagger)+2\phi_\Delta)$

In [ ]:
import dynamiqs as dq
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
import matplotlib.colors as mcolors
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.6 * 2 * np.pi
w_b = 6.6 * 2 * np.pi

phi_a = 0.11
phi_b = 0.2

E_J = 26 * 2 * np.pi 
ELp = 7250*  2 * np.pi
E_2J = (E_J**2 + (0.1*E_J)**2)/2/ELp

# drive/pump base params
epsilon_0_base = 0.08
g2 = sp.special.jv(1, epsilon_0_base) * E_J * phi_a**2 * phi_b

In [ ]:
n_a = 20
n_b = 10
a = dq.destroy(n_a)
b = dq.destroy(n_b)
n_a_tot = dq.tensor(dq.dag(a)@a,dq.eye(n_b))
n_b_tot = dq.tensor(dq.eye(n_a), dq.dag(b)@b)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b))

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 201)
transitions_list = []
alphas_list = []
matrix_elements_list = []

# Build photon number operator in the full Hilbert space
# (same structure as varphi_a_op_tot — adapt to your basis)
n_op = n_a_tot + n_b_tot   # or whichever number op is relevant
n_op_np = n_op.to_numpy()

drive_op_np = (1j*dq.tensor(dq.eye(n_a), dq.dag(b)-b)).to_numpy()

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    idx = np.argsort(np.real(evals))
    evecs = evecs[:, idx]   # sort eigenvectors consistently with eigenvalues

    # Mean photon number <n> for each eigenstate
    n_means = np.array([
        np.real(evecs[:, k].conj() @ n_op_np @ evecs[:, k])
        for k in range(10)
    ])

    transitions = np.real(evals[idx]) - np.real(evals[idx[0]])
    transitions_list.append(transitions[:10])
    alphas_list.append(n_means)
    ground = evecs[:, 0]
    matrix_elements_list.append(np.array([
        np.abs(evecs[:,k].conj() @ drive_op_np @ ground) ** 2
        for k in range(10)])
    )

transitions_arr = np.array(transitions_list)   # shape (101, 10)
alphas_arr      = np.array(alphas_list)        # shape (101, 10)
matrix_elements_arr = np.array(matrix_elements_list)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Normalize across all phi and all bands
drive_op_min = matrix_elements_arr.min()
drive_op_max = matrix_elements_arr.max()
alphas_norm = (matrix_elements_arr - drive_op_min) / (drive_op_max - drive_op_min + 1e-12)

fig, ax = plt.subplots(figsize=(8, 5))

for k in range(1, 10):   # skip ground state (always 0)
    for i in range(len(phi_delta_list) - 1):
        ax.plot(
            phi_delta_list[i:i+2],
            transitions_arr[i:i+2, k]/ (2*np.pi),
            color='steelblue',
            alpha=float(alphas_norm[i, k]),
            linewidth=1.5,
            solid_capstyle='round',
        )

ax.set_xlabel(r'$\phi_\delta$')
ax.set_ylabel('Energy ' + r"$\omega/2\pi$"+' (GHz)')
ax.set_title('Spectroscopy \n transparency ∝ drive operator matrix element (ground to excited)')

# Colorbar as a proxy for alpha scale
sm = plt.cm.ScalarMappable(
    cmap=mcolors.LinearSegmentedColormap.from_list('alpha_cmap', ['white', 'steelblue']),
    norm=plt.Normalize(vmin=drive_op_min, vmax=drive_op_max)
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r'$| \langle k| i(b^\dagger - b)|0 \rangle| ^2$')
plt.xlim(np.pi-0.5, np.pi+0.5)
plt.ylim(3.5, 7.5)
plt.tight_layout()
plt.show()

In [ ]:
for i in range(1, 10):
    plt.plot(phi_delta_list, transitions_arr[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency (GHz)')
plt.show()

## Shifting a few degrees from zero for $\phi_\Sigma$

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 101)
transitions_list = []

phi_sigma = 3*np.pi/180
for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * np.cos(phi_sigma) * dq.cosm(phi_op) + 2 * E_J*0.1 * np.sin(phi_sigma) * dq.sinm(phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list_few_degrees = np.array(transitions_list)

In [ ]:
fig, axes = plt.subplots(1, 2)
ax_1, ax_2 = axes
for i in range(1, 10):
    ax_1.plot(phi_delta_list, transitions_list_few_degrees[:, i] / (2*np.pi))
    ax_2.plot(phi_delta_list, 100*(transitions_list_few_degrees[:, i]-transitions_list_bare[:, i]) / transitions_list_bare[:, i])

ax_1.set_xlabel(r'$\phi_\Delta$')
ax_1.set_ylabel('Transition freq. (GHz)')

ax_2.set_xlabel(r'$\phi_\Delta$')
ax_2.set_ylabel('Normalized Gap in \n Transition freq. (%)')
plt.tight_layout()
plt.show()

# Adding the $\cos(2\phi)$-term
$$H = \omega_a a^\dagger a + \omega_b b^\dagger b - 2E_J\cos(\phi_\Sigma)\cos(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + 2\Delta E_J\sin(\phi_\Sigma)\sin(\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\phi_\Delta) + E_{2J}\cos(2\phi_\Sigma)\cos(2\varphi_a (a+a^\dagger) + 2\varphi_b (b+b^\dagger)+2\phi_\Delta),$$
with $E_{2J} = (E_{J_1}^2+E_{J_2})/4\epsilon_L = ((E_J+\Delta E_J)^2+(E_J-\Delta E_J)^2)/4\epsilon_L = (E_J^2+\Delta E_J^2)/2\epsilon_L$

In [ ]:
phi_delta_list = np.linspace(0, 2*np.pi, 101)
transitions_list = []

for phi_delta in phi_delta_list:
    phi_op = varphi_a_op_tot + varphi_b_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op) + E_2J * dq.cosm(2*phi_op)
    H = H.to_numpy()

    evals, evecs = np.linalg.eigh(H)
    evals = np.real(evals)
    evals = np.sort(evals)

    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:10])

transitions_list_cos_2phi = np.array(transitions_list)

In [ ]:
fig, axes = plt.subplots(1, 2)
ax_1, ax_2 = axes
for i in range(1, 10):
    ax_1.plot(phi_delta_list, transitions_list_cos_2phi[:, i] / (2*np.pi))
    ax_2.plot(phi_delta_list, 100*(transitions_list_cos_2phi[:, i]-transitions_list_bare[:, i]) / transitions_list_bare[:, i])

ax_1.set_xlabel(r'$\phi_\Delta$')
ax_1.set_ylabel('Transition freq. (GHz)')

ax_2.set_xlabel(r'$\phi_\Delta$')
ax_2.set_ylabel('Normalized Gap in \n Transition freq. (%)')
plt.tight_layout()
plt.show()

# We now add a third mode in the ATS 
This mode will have high frequency $\omega_c/2\pi=15$ GHz, and a large $\varphi_{zpf}$, let's take $\varphi_{c} = 10\varphi_a$

$$H = \omega_a a^\dagger a + \omega_b b^\dagger b + \omega_c c^\dagger c - 2E_J\cos\bigl[\varphi_a (a+a^\dagger) + \varphi_b (b+b^\dagger)+\varphi_c (c+c^\dagger)+\phi_\Delta\bigr],$$

In [ ]:
#w_c = 15 * 2 * np.pi
#phi_c = 0.3
w_a = 3.823578966821696*2*np.pi#3.85
w_b = 6.697567050165361*2*np.pi#6.761
w_c = 15.0*2*np.pi
# Ej = 27.148
# Delta_Ej = 0.316
# phi_zpf_mem = 0.139
# phi_zpf_buf = 0.204
# phi_zpf_ats = 0.1
Ej = 22.4
phi_a = 0.071
phi_b = 0.171
phi_c = 0.3


n_a = 12
n_b = 6
n_c = 5
a = dq.destroy(n_a)
b = dq.destroy(n_b)
c = dq.destroy(n_c)
varphi_a_op_tot = dq.tensor(phi_a*(a + a.dag()), dq.eye(n_b), dq.eye(n_c))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), phi_b*(b + b.dag()), dq.eye(n_c))
varphi_c_op_tot = dq.tensor(dq.eye(n_a), dq.eye(n_b), phi_c*(c + c.dag()))

H_base = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b), dq.eye(n_c)) \
    + dq.tensor(dq.eye(n_a), dq.eye(n_b),  w_c * dq.number(n_c))

In [ ]:
phi_delta_list = np.linspace(2.5, 3.75, 60)
transitions_list = []
matrix_elements_list = []
drive_op_np = (1j*dq.tensor(dq.eye(n_a), dq.dag(b)-b, dq.eye(n_c))).to_numpy()#(1j*dq.tensor(dq.eye(n_a), dq.eye(n_b), dq.dag(c)-c)).to_numpy()#
n_state = 4
for phi_delta in phi_delta_list:
    print(phi_delta/2/np.pi)
    phi_op = varphi_a_op_tot + varphi_b_op_tot + varphi_c_op_tot + phi_delta * dq.eye_like(varphi_a_op_tot)
    H = H_base - 2 * E_J * dq.cosm(phi_op)
    H = H.to_numpy()

    evals, evecs = sp.sparse.linalg.eigsh(H, k=n_state, which='SM', tol = 0, maxiter=1e6)
    evals = np.real(evals)
    evals = np.sort(evals)
    transitions = evals - evals[0]   # spectroscopy from ground state
    transitions_list.append(transitions[:n_state])
    ground = evecs[:, 0]
    matrix_elements_list.append(np.array([
        np.abs(evecs[:,k].conj() @ drive_op_np @ ground) ** 2
        for k in range(n_state)])
    )

transitions_arr = np.array(transitions_list)
matrix_elements_arr = np.array(matrix_elements_list)

In [ ]:
best_idx = np.argmax(matrix_elements_arr, axis=1) 
best_idx[:10] = 2*np.ones(10)         # shape (n,)
best_idx[-15:] = 2*np.ones(15)         # shape (n,)
transitions_filtered = transitions_arr[np.arange(len(transitions_arr)), best_idx]  # shape (n,)

In [ ]:
# Normalize across all phi and all bands
drive_op_min = matrix_elements_arr.min()
drive_op_max = matrix_elements_arr.max()
alphas_norm = (matrix_elements_arr - drive_op_min) / (drive_op_max - drive_op_min + 1e-12)

fig, ax = plt.subplots(figsize=(8, 5))

for k in range(2, n_state):   # skip ground state (always 0)
    for i in range(len(phi_delta_list) - 1):
        ax.plot(
            phi_delta_list[i:i+2],
            transitions_arr[i:i+2, k]/ (2*np.pi),
            color='steelblue',
            alpha=float(alphas_norm[i, k]),
            linewidth=1.5,
            solid_capstyle='round',
        )
plt.scatter(phi_delta_list, transitions_filtered/2/np.pi)
ax.set_xlabel(r'$\phi_\delta$')
ax.set_ylabel('Energy ' + r"$\omega/2\pi$"+' (GHz)')
ax.set_title('Spectroscopy \n transparency ∝ drive operator matrix element (ground to excited)')

# Colorbar as a proxy for alpha scale
sm = plt.cm.ScalarMappable(
    cmap=mcolors.LinearSegmentedColormap.from_list('alpha_cmap', ['white', 'steelblue']),
    norm=plt.Normalize(vmin=drive_op_min, vmax=drive_op_max)
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label=r'$| \langle k| i(b^\dagger - b)|0 \rangle| ^2$')

plt.tight_layout()
plt.show()

In [ ]:
for i in range(1, n_state):
    plt.plot(phi_delta_list, transitions_arr[:, i] / (2*np.pi))

plt.xlabel(r'$\phi_\Delta$')
plt.ylabel('Transition frequency (GHz)')
plt.show()

In [ ]:
x_1 = [2.5031078610603292, 2.5360146252285194, 2.579890310786106, 2.6228519195612434, 2.6575868372943328, 2.693235831809872, 2.729798903107861, 2.779159049360146, 2.8148080438756855, 2.850457038391225, 2.8851919561243147, 2.913528336380256, 2.951005484460695, 2.976599634369287, 3.0095063985374773, 3.2755027422303473, 3.2956124314442414, 3.3212065813528335, 3.3431444241316273, 3.36782449725777, 3.396160877513711, 3.4217550274223036, 3.450091407678245, 3.4784277879341867, 3.504936014625229, 3.5277879341864717, 3.5588665447897623, 3.5881170018281536, 3.627422303473492, 3.663985374771481, 3.7023765996343694, 3.7389396709323584]
y_1 = [7.391752577319588, 7.367010309278351, 7.325773195876288, 7.284536082474227, 7.2329896907216495, 7.179381443298969, 7.1319587628865975, 7.063917525773196, 7.014432989690722, 6.958762886597938, 6.907216494845361, 6.8391752577319584, 6.756701030927835, 6.6969072164948455, 6.616494845360824, 6.591752577319587, 6.645360824742268, 6.701030927835052, 6.748453608247423, 6.818556701030928, 6.874226804123712, 6.91958762886598, 6.969072164948454, 7.0123711340206185, 7.05360824742268, 7.08041237113402, 7.115463917525773, 7.156701030927835, 7.216494845360825, 7.268041237113402, 7.309278350515464, 7.348453608247423]
x_2 = [3.076234003656307, 3.0926873857404025, 3.110054844606947, 3.1265082266910422, 3.173126142595978, 3.1850091407678245, 3.20146252285192, 3.2151736745886654]
y_2 = [7.367010309278351, 7.28659793814433, 7.191752577319588, 7.115463917525773, 7.119587628865979, 7.202061855670103, 7.276288659793814, 7.3546391752577325]
plt.scatter(x_1, y_1)
plt.scatter(x_2, y_2)
plt.show()